# 미션 02. seed별 로컬 LLM 성공률 평가기

한 번의 성공이나 실패로 모델의 능력을 단정하지 않고, 여러 seed에서 **핵심 패턴·엄격한 형식·주제 준수율**을 나누어 측정합니다. 마지막에는 같은 평가 로직을 Streamlit 대시보드에서 사용합니다.

## 완료 조건

- 자동 평가가 할 수 있는 판단과 사람이 해야 하는 판단을 구분한다.
- 동일 설정을 여러 seed로 실행하고 성공률을 계산한다.
- 원본 프롬프트와 few-shot 프롬프트의 성공률을 비교한다.
- Streamlit 앱에서 생성 결과를 검토하고 주제 준수 여부를 직접 평가한다.

## 0. 이번 미션의 평가 개념

| 구분 | 질문 | 이번 미션의 측정 방법 |
|---|---|---|
| 능력 보유 | 올바른 이행시 경로가 존재하는가? | seed 중 한 번이라도 핵심 패턴 성공 |
| 지시 준수 | 요구한 조건을 모두 지키는가? | 형식과 주제를 항목별 평가 |
| 안정성 | seed가 달라도 계속 성공하는가? | 여러 seed의 성공률 계산 |

> seed는 모델에 새로운 능력을 추가하지 않습니다. 샘플링에서 선택되는 생성 경로를 바꾸므로, 여러 seed의 결과는 모델이 가진 출력 분포를 관찰하는 표본입니다.

## 1. 공용 평가 모듈 준비

노트북과 Streamlit이 동일한 생성·평가 함수를 사용하도록 `llm_evaluator.py`에서 가져옵니다. 실행 위치가 프로젝트 루트인지 `day01`인지 모두 처리합니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch

module_dir = Path.cwd()
if not (module_dir / "llm_evaluator.py").exists():
    module_dir = Path.cwd() / "day01"
if not (module_dir / "llm_evaluator.py").exists():
    raise FileNotFoundError("day01/llm_evaluator.py를 찾을 수 없습니다.")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from llm_evaluator import (
    DEFAULT_MODEL_ID,
    evaluate_acrostic,
    load_model,
    run_seed_experiment,
    success_rates,
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

/home/student/llm-practice/c5-slm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: /home/student/llm-practice/c5-slm/.venv/bin/python
PyTorch: 2.13.0+cu130
CUDA: True


## 2. 생성 전에 평가 기준부터 검증하기

`evaluate_acrostic()`은 다음 세 값을 자동으로 확인합니다.

- `core_pattern_pass`: 구절들이 순서대로 `점 → 심`으로 시작하는가?
- `strict_format_pass`: 불필요한 줄 없이 정확히 두 줄이고 `점 → 심`으로 시작하는가?
- `keyword_hit`: 점심 관련 단어가 포함됐는가?

키워드 포함은 의미 이해가 아니므로 최종 주제 평가는 사람이 합니다.

In [2]:
sample_answers = {
    "전체 성공 후보": "점: 점심 메뉴를 고르고\n심: 심장이 설레는 식사 시간",
    "핵심 패턴만 성공": "점심은 소리가 빛나는 곳, 심장이 힘들어하는 곳.",
    "요청 반복": "점심을 주제로 한 이행시\n점심을 주제로 한 이행시",
}

for label, answer in sample_answers.items():
    print(f"\n[{label}]")
    print(answer)
    print(evaluate_acrostic(answer))


[전체 성공 후보]
점: 점심 메뉴를 고르고
심: 심장이 설레는 식사 시간
{'core_pattern_pass': True, 'strict_format_pass': True, 'keyword_hit': True, 'matched_keywords': '점심, 식사, 메뉴'}

[핵심 패턴만 성공]
점심은 소리가 빛나는 곳, 심장이 힘들어하는 곳.
{'core_pattern_pass': True, 'strict_format_pass': False, 'keyword_hit': True, 'matched_keywords': '점심'}

[요청 반복]
점심을 주제로 한 이행시
점심을 주제로 한 이행시
{'core_pattern_pass': False, 'strict_format_pass': False, 'keyword_hit': False, 'matched_keywords': ''}


### 예제 결과 해석

**Q1. `핵심 패턴만 성공`은 왜 전체 성공이 아닌가?**
> `점 → 심` 순서는 만들었지만 정확히 두 줄이라는 형식을 지키지 않았고, 내용도 점심 식사와 의미상 연결되지 않는다. 패턴 생성 능력과 전체 지시 성공을 구분해야 한다.

**Q2. `keyword_hit=True`이면 주제 성공으로 확정할 수 있는가?**
> **확정할 수 없다.** 요청을 그대로 반복하거나 `점심`이라는 단어만 우연히 포함해도 통과할 수 있다.  
> 자동 키워드 평가는 사람이 볼 결과를 좁혀주는 보조 지표다.

## 3. 모델 준비

미션 01과 동일하게 `Qwen/Qwen3-0.6B`를 CUDA FP16으로 불러옵니다. CUDA가 보이지 않으면 CPU로 우회하지 않고 오류를 냅니다.

In [3]:
tokenizer, model = load_model(DEFAULT_MODEL_ID)
print("준비 완료:", DEFAULT_MODEL_ID)
print("사용 GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 305.67it/s]


준비 완료: Qwen/Qwen3-0.6B
사용 GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 4. 비교할 프롬프트와 생성 설정

원본 프롬프트와 완성 예시를 포함한 few-shot 프롬프트를 같은 생성 설정으로 비교합니다. 한 번에 프롬프트만 바꿔야 차이의 원인을 해석할 수 있습니다.

In [4]:
original_prompt = (
    "점심을 주제로 '점'과 '심'으로 시작하는 이행시를 지어줘. "
    "각 행은 20자 이내로 쓰고, 설명 없이 이행시만 답해."
)

few_shot_prompt = """
한국어 2행시를 작성하세요.

예시:
제시어: 바다
바: 바람이 살며시 불어오고
다: 다정한 파도가 반겨준다

새 과제:
제시어: 점심
주제: 즐거운 점심시간
규칙: 점과 심으로 시작하는 두 줄만 출력하고 제목이나 설명은 쓰지 마세요.
답변:
""".strip()

options = {
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 20,
    "repetition_penalty": 1.0,
    "max_new_tokens": 80,
}

## 5. 단일 seed로 전체 흐름 확인

먼저 한 번만 실행해 생성과 평가가 연결되는지 확인합니다. 단일 결과는 동작 확인용이며 성능 결론으로 사용하지 않습니다.

In [5]:
single_rows = run_seed_experiment(
    model,
    tokenizer,
    few_shot_prompt,
    seeds=[2026],
    generation_options=options,
)
single_rows[0]

{'seed': 2026,
 'answer': '점심',
 'new_tokens': 3,
 'seconds': 1.27,
 'tok_per_s': 2.4,
 'core_pattern_pass': False,
 'strict_format_pass': False,
 'keyword_hit': True,
 'matched_keywords': '점심',
 'manual_topic_pass': False}

## 6. 여러 seed로 성공률 측정

처음에는 10회로 실험합니다. 표본이 적으므로 이 숫자를 모델의 절대 성능으로 일반화하지 말고, 프롬프트 간 상대 비교에 사용합니다.

In [6]:
seeds = list(range(10))
few_shot_rows = run_seed_experiment(
    model,
    tokenizer,
    few_shot_prompt,
    seeds=seeds,
    generation_options=options,
)
few_shot_df = pd.DataFrame(few_shot_rows)
few_shot_df[[
    "seed", "answer", "core_pattern_pass", "strict_format_pass",
    "keyword_hit", "seconds", "tok_per_s",
]]

,seed,answer,core_pattern_pass,strict_format_pass,keyword_hit,seconds,tok_per_s
0,0,점심,False,False,True,0.13,23.4
1,1,점심,False,False,True,0.09,33.4
2,2,점심,False,False,True,0.09,32.3
3,3,점심,False,False,True,0.10,31.1
4,4,점심,False,False,True,0.11,26.6
5,5,점심,False,False,True,0.09,32.3
6,6,점심,False,False,True,0.11,27.8
7,7,점심,False,False,True,0.11,28.1
8,8,점심,False,False,True,0.13,23.9
9,9,점심,False,False,True,0.13,23.8


## 7. 사람이 주제를 평가한 뒤 성공률 계산

표의 답변을 읽고 실제로 점심 주제를 유지한 seed 번호를 집합에 넣으세요. `keyword_hit` 값을 그대로 복사하지 말고 의미를 직접 판단합니다.

In [7]:
# TODO: 점심 주제를 실제로 지킨 seed를 직접 기록하세요. 예: {0, 3, 7}
manual_topic_pass_seeds = set()

few_shot_df["manual_topic_pass"] = few_shot_df["seed"].isin(manual_topic_pass_seeds)
rates = success_rates(few_shot_df.to_dict(orient="records"))
pd.Series(rates, name="성공률(%)")

core_pattern_rate     0.0
strict_format_rate    0.0
manual_topic_rate     0.0
overall_rate          0.0
Name: 성공률(%), dtype: float64

## 8. 원본 프롬프트와 few-shot 프롬프트 비교

동일한 다섯 seed와 생성 설정을 사용합니다. 실행 시간이 부담되면 `comparison_seeds`를 세 개로 줄여도 됩니다.

In [8]:
comparison_seeds = [0, 1, 2, 3, 4]
comparison_rows = []

for prompt_name, prompt in [
    ("원본", original_prompt),
    ("few-shot", few_shot_prompt),
]:
    rows = run_seed_experiment(
        model, tokenizer, prompt, comparison_seeds, generation_options=options
    )
    for row in rows:
        comparison_rows.append({"prompt": prompt_name, **row})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.groupby("prompt")[[
    "core_pattern_pass", "strict_format_pass", "keyword_hit"
]].mean().mul(100).round(1)

,core_pattern_pass,strict_format_pass,keyword_hit
prompt,,,
few-shot,0.0,0.0,100.0
원본,0.0,0.0,60.0


### 결과 기록

**Q1. 어느 프롬프트의 핵심 패턴 성공률이 높았는가?**
> **두 프롬프트 모두 `0%`로 차이가 없었다.**  
> 동일한 seed 5개를 비교한 결과, 원본과 few-shot 모두 `core_pattern_pass=0%`, `strict_format_pass=0%`였다. 따라서 이번 실행에서는 few-shot 예시가 `점 → 심` 패턴이나 정확한 두 줄 형식을 끌어내지 못했다.

**Q2. few-shot 프롬프트가 모든 seed를 성공시켰는가?**
> **아니다. 비교 실험의 5개 seed와 별도 반복 실험의 10개 seed 모두 실패했다.**  
> 10회 반복에서는 모든 답변이 `점심` 한 단어로 끝났다. 그래서 `keyword_hit`는 100%였지만 첫 구절을 `점`, 다음 구절을 `심`으로 만드는 핵심 패턴과 정확히 두 줄을 출력하는 형식은 모두 0%였다.  
> 이는 반복이나 토큰 분할 실패가 아니라 **제시어만 출력하고 너무 일찍 생성을 끝낸 실패 유형**이다.  
> 현재의 예시 하나만으로는 0.6B 모델이 요구한 출력 구조를 안정적으로 이어 쓰게 만들지 못했다.

**Q3. `keyword_hit=100%`를 주제 성공률 100%라고 해석할 수 있는가?**
> **해석할 수 없다.**  
> `점심` 한 단어만 출력해도 키워드 검사를 통과하지만, 이행시 내용은 전혀 생성되지 않았다.  
> 수동 평가에서 통과 seed를 하나도 선택하지 않은 것이 타당하며, 실제 주제 성공률과 전체 성공률은 모두 0%다.  
> 이 결과는 키워드 검사가 의미 평가가 아니라 보조 지표라는 한계를 잘 보여준다.

**Q4. 이번 성공률을 모델의 절대 성능이라고 할 수 있는가?**
> **아니다.** seed와 프롬프트 수가 적고 평가 주제도 하나뿐이며, 미션 01의 seed 2026에서는 이행시 핵심 패턴을 생성한 사례도 있었다.  
> 이번 `0%`는 모델에 능력이 전혀 없다는 뜻이 아니라, **현재 few-shot 프롬프트와 생성 설정에서는 그 능력을 안정적으로 끌어내지 못했다**는 결과다.

## 9. Streamlit 대시보드로 확장

`streamlit_app.py`는 이 노트북과 같은 `llm_evaluator.py`를 사용합니다. 슬라이더로 생성 옵션과 seed 수를 바꾸고, 결과 표에서 `사람 주제 평가`를 직접 체크하면 전체 성공률이 갱신됩니다.

### Streamlit 설치 확인

현재 프로젝트의 `.venv`에는 Streamlit이 설치되어 있습니다. 아래 셀로 노트북 커널에서도 같은 패키지를 보는지 확인합니다. 다른 환경에서 다시 구성할 때는 프로젝트 루트에서 `uv pip install --python .venv/bin/python streamlit`을 실행하세요.

In [9]:
import streamlit
print("Streamlit:", streamlit.__version__)

Streamlit: 1.62.0


### 앱 실행 전 주의

노트북과 Streamlit은 별도 프로세스이므로 모델을 각각 GPU에 올립니다. **먼저 이 노트북 커널을 종료해 VRAM을 비운 뒤**, 프로젝트 루트의 터미널에서 실행하세요.

```bash
.venv/bin/python -m streamlit run day01/streamlit_app.py
```

앱에서 확인할 것:

1. 같은 설정에서 seed 수를 늘리면 성공률이 어떻게 변하는가?
2. 자동 `keyword_hit`와 사람의 주제 평가가 다른 사례가 있는가?
3. temperature를 바꾸면 핵심 패턴 성공률과 답변 다양성이 함께 어떻게 변하는가?